In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
import networkx as nx
import json
import pickle
import warnings
from collections import defaultdict
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")

df         = pd.read_parquet("/kaggle/input/notebooks/anymansince2005/user-modeling/arxiv_ml_final.parquet")
df         = df.reset_index(drop=True)
embeddings = np.load("/kaggle/input/notebooks/anymansince2005/semantic-embedding/embeddings.npy")
faiss_index= faiss.read_index("/kaggle/input/notebooks/anymansince2005/semantic-embedding/faiss_index.bin")
with open("/kaggle/input/notebooks/anymansince2005/citation-graphs/citation_graph.pkl", "rb") as f:
    G = pickle.load(f)
model      = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
class UserProfile:

    PIN_WEIGHT    = 3.0
    RECENCY_DECAY = 0.95   

    def __init__(self, user_id: str):
        self.user_id         = user_id
        self.read_indices    = []     
        self.pinned_indices  = set()
        self.category_counts = defaultdict(int)
        self.profile_vector  = None
        self.created_at      = pd.Timestamp.now()
        self.updated_at      = pd.Timestamp.now()

    def add_read(self, paper_idx: int, df: pd.DataFrame):
        if paper_idx not in self.read_indices:
            self.read_indices.append(paper_idx)
            for cat in df.iloc[paper_idx]["categories"]:
                self.category_counts[cat] += 1
            self.updated_at = pd.Timestamp.now()

    def pin_paper(self, paper_idx: int, df: pd.DataFrame):
        self.pinned_indices.add(paper_idx)
        if paper_idx not in self.read_indices:
            self.add_read(paper_idx, df)

    def build_profile_vector(self, embeddings: np.ndarray) -> np.ndarray:
        if not self.read_indices:
            return None

        vecs    = []
        weights = []
        n       = len(self.read_indices)

        for rank, idx in enumerate(self.read_indices):
            recency_w = self.RECENCY_DECAY ** (n - 1 - rank)
            pin_w     = self.PIN_WEIGHT if idx in self.pinned_indices else 1.0
            weight    = recency_w * pin_w

            vecs.append(embeddings[idx])
            weights.append(weight)

        weights = np.array(weights)
        weights = weights / weights.sum()              

        profile = np.average(vecs, axis=0, weights=weights)
        profile = profile / np.linalg.norm(profile)    

        self.profile_vector = profile.astype(np.float32)
        return self.profile_vector

    def top_categories(self, top_n: int = 3) -> list:
        return sorted(
            self.category_counts.items(),
            key=lambda x: -x[1]
        )[:top_n]

    def summary(self) -> dict:
        return {
            "user_id"       : self.user_id,
            "papers_read"   : len(self.read_indices),
            "papers_pinned" : len(self.pinned_indices),
            "top_categories": self.top_categories(3),
            "has_profile"   : self.profile_vector is not None,
        }

In [4]:
with open("/kaggle/input/notebooks/anymansince2005/user-modeling/user_profiles.pkl", "rb") as f:
    profiles = pickle.load(f)

with open("/kaggle/input/notebooks/anymansince2005/user-modeling/all_metrics.json") as f:
    all_metrics = json.load(f)

print(f"Loaded {len(df):,} papers")
print(f"Columns  : {df.columns.tolist()}")
print(f"Profiles : {list(profiles.keys())}")

Loaded 110 papers
Columns  : ['paper_id', 'title', 'abstract', 'categories', 'date', 'corpus', 'graph_in_degree', 'graph_pagerank', 'graph_authority', 'graph_hub', 'pr_norm', 'auth_norm', 'ind_norm', 'graph_score']
Profiles : ['nlp_researcher', 'cv_researcher', 'new_user']


In [5]:
def maximal_marginal_relevance(
    query_vec       : np.ndarray,
    candidate_idxs  : list[int],
    embeddings      : np.ndarray,
    top_n           : int   = 10,
    lambda_param    : float = 0.60,
) -> list[tuple[int, float]]:
    query_vec  = query_vec.flatten()
    cand_vecs  = embeddings[candidate_idxs]
    rel_scores = cand_vecs @ query_vec

    selected      = []
    selected_vecs = []
    remaining     = list(zip(candidate_idxs, rel_scores))

    while remaining and len(selected) < top_n:
        if not selected_vecs:
            best_idx = int(np.argmax([r for _, r in remaining]))
        else:
            sel_mat    = np.array(selected_vecs)
            mmr_scores = []
            for cand_idx, rel in remaining:
                cand_vec        = embeddings[cand_idx]
                sim_to_selected = float((sel_mat @ cand_vec).max())
                mmr_scores.append(lambda_param * rel - (1 - lambda_param) * sim_to_selected)
            best_idx = int(np.argmax(mmr_scores))

        best_paper_idx, best_rel = remaining.pop(best_idx)
        selected.append((best_paper_idx, round(float(best_rel), 4)))
        selected_vecs.append(embeddings[best_paper_idx])

    return selected


def rerank_with_mmr(
    query_vec    : np.ndarray,
    initial_recs : pd.DataFrame,
    embeddings   : np.ndarray,
    top_n        : int   = 10,
    lambda_param : float = 0.60,
) -> pd.DataFrame:
    candidate_idxs = list(
        initial_recs.index if "cand_idx" not in initial_recs.columns
        else initial_recs["cand_idx"]
    )
    mmr_results   = maximal_marginal_relevance(
        query_vec, candidate_idxs, embeddings, top_n, lambda_param
    )
    reranked_idxs   = [r[0] for r in mmr_results]
    reranked_scores = [r[1] for r in mmr_results]

    rec_df = df.iloc[reranked_idxs][
        ["paper_id", "title", "categories", "date", "graph_pagerank"]
    ].copy()
    rec_df["relevance_score"] = reranked_scores
    rec_df["mmr_rank"]        = range(1, len(rec_df) + 1)
    return rec_df.reset_index(drop=True)

In [6]:
def apply_recency_boost(
    recs            : pd.DataFrame,
    half_life_years : float = 3.0,
    boost_weight    : float = 0.15,
) -> pd.DataFrame:
    recs         = recs.copy()
    now          = pd.Timestamp.now()
    recs["date"] = pd.to_datetime(recs["date"], errors="coerce")

    age_years       = (now - recs["date"]).dt.days / 365.25
    recs["recency"] = np.exp(-np.log(2) * age_years / half_life_years).clip(0, 1).round(4)

    rel      = recs["relevance_score"]
    rel_norm = (rel - rel.min()) / (rel.max() - rel.min() + 1e-9)

    recs["final_score"] = (
        (1 - boost_weight) * rel_norm +
        boost_weight       * recs["recency"]
    ).round(4)

    return recs.sort_values("final_score", ascending=False).reset_index(drop=True)

def category_diversity(recs: pd.DataFrame) -> float:
    all_cats = [cat for cats in recs["categories"] for cat in cats]
    unique   = len(set(all_cats))
    total    = len(all_cats)
    return round(unique / total if total > 0 else 0.0, 4)


def intra_list_diversity(
    recs        : pd.DataFrame,
    embeddings  : np.ndarray,
    rec_indices : list[int],
) -> float:
    if len(rec_indices) < 2:
        return 0.0
    vecs  = embeddings[rec_indices]
    sims  = vecs @ vecs.T
    n     = len(rec_indices)
    upper = sims[np.triu_indices(n, k=1)]
    return round(float(1 - upper.mean()), 4)


def novelty_score(recs: pd.DataFrame) -> float:
    pr_values = recs["graph_pagerank"].values
    novelty   = 1 - (pr_values - pr_values.min()) / (pr_values.max() - pr_values.min() + 1e-9)
    return round(float(novelty.mean()), 4)


def compute_diversity_report(
    recs        : pd.DataFrame,
    embeddings  : np.ndarray,
    rec_indices : list[int],
    label       : str = "",
) -> dict:
    return {
        "label"               : label,
        "category_diversity"  : category_diversity(recs),
        "intra_list_diversity": intra_list_diversity(recs, embeddings, rec_indices),
        "novelty_score"       : novelty_score(recs),
        "n_unique_categories" : len(set(cat for cats in recs["categories"] for cat in cats)),
        "n_recommendations"   : len(recs),
    }

In [7]:
def fairness_audit(
    df                : pd.DataFrame,
    embeddings        : np.ndarray,
    faiss_index,
    sample_size       : int   = 300,
    top_n             : int   = 10,
    pagerank_threshold: float = None,
) -> dict:
    if pagerank_threshold is None:
        pagerank_threshold = df["graph_pagerank"].quantile(0.90)

    corpus_mean_pr = df["graph_pagerank"].mean()
    recent_cutoff  = pd.Timestamp.now() - pd.DateOffset(years=2)
    df["date"]     = pd.to_datetime(df["date"], errors="coerce")

    rec_pr_vals    = []
    top10_pct_hits = []
    recent_hits    = []

    indices = np.random.choice(len(df), size=sample_size, replace=False)

    for idx in indices:
        query_vec   = embeddings[idx].reshape(1, -1).astype(np.float32)
        _, rec_idxs = faiss_index.search(query_vec, top_n + 1)
        rec_idxs    = [int(i) for i in rec_idxs[0] if i != idx][:top_n]
        recs_subset = df.iloc[rec_idxs]

        rec_pr_vals.extend(recs_subset["graph_pagerank"].tolist())
        top10_pct_hits.append(
            (recs_subset["graph_pagerank"] >= pagerank_threshold).sum() / top_n
        )
        recent_hits.append(
            (recs_subset["date"] >= recent_cutoff).sum() / top_n
        )

    mean_rec_pr   = float(np.mean(rec_pr_vals))
    pr_bias_ratio = round(mean_rec_pr / (corpus_mean_pr + 1e-12), 3)

    report = {
        "corpus_mean_pagerank" : round(corpus_mean_pr, 8),
        "rec_mean_pagerank"    : round(mean_rec_pr, 8),
        "popularity_bias_ratio": pr_bias_ratio,
        "top10pct_exposure"    : round(float(np.mean(top10_pct_hits)), 4),
        "recency_exposure_2yr" : round(float(np.mean(recent_hits)), 4),
        "sample_size"          : sample_size,
    }

    print("\n" + "=" * 60)
    print("FAIRNESS AUDIT REPORT")
    print("=" * 60)
    print(f"\n  Corpus mean PageRank      : {report['corpus_mean_pagerank']:.8f}")
    print(f"  Rec mean PageRank         : {report['rec_mean_pagerank']:.8f}")
    print(f"  Popularity bias ratio     : {pr_bias_ratio}x")
    print(f"    (>1.0 = over-recommends popular papers)")
    print(f"\n  Top-10% PageRank exposure : {report['top10pct_exposure']*100:.1f}%")
    print(f"    (% of recs from top-10% most cited papers)")
    print(f"\n  Recency exposure (2yr)    : {report['recency_exposure_2yr']*100:.1f}%")
    print(f"    (% of recs from last 2 years)")
    return report

In [8]:
def generate_explanation(
    query_idx : int,
    rec_idx   : int,
    df        : pd.DataFrame,
    G         : nx.DiGraph,
    sem_score : float,
) -> str:
    rec_row = df.iloc[rec_idx]

    if G.has_edge(query_idx, rec_idx) or G.has_edge(rec_idx, query_idx):
        return "Frequently cited with your paper"

    pr_threshold = df["graph_pagerank"].quantile(0.85)
    if sem_score > 0.80 and rec_row["graph_pagerank"] >= pr_threshold:
        return "Highly influential paper in this area"

    if sem_score > 0.75:
        return "Closely matches your research interest"

    query_cats  = set(df.iloc[query_idx]["categories"])
    rec_cats    = set(rec_row["categories"])
    shared_cats = query_cats & rec_cats
    if shared_cats:
        return f"Trending in {list(shared_cats)[0]}"

    return "Related to your reading history"


def attach_explanations(
    query_idx : int,
    recs      : pd.DataFrame,
    df        : pd.DataFrame,
    G         : nx.DiGraph,
) -> pd.DataFrame:
    recs      = recs.copy()
    score_col = next(
        (c for c in ["relevance_score", "blended_score", "similarity_score"]
         if c in recs.columns), None
    )
    why_tags  = []

    for _, row in recs.iterrows():
        match = df[df["paper_id"] == row["paper_id"]].index
        if match.empty:
            why_tags.append("Related to your reading history")
            continue
        rec_idx   = int(match[0])
        sem_score = float(row[score_col]) if score_col else 0.0
        why_tags.append(generate_explanation(query_idx, rec_idx, df, G, sem_score))

    recs["why_recommended"] = why_tags
    return recs

In [9]:
def full_recommendation_pipeline(
    query_idx       : int,
    embeddings      : np.ndarray,
    faiss_index,
    df              : pd.DataFrame,
    G               : nx.DiGraph,
    top_n           : int   = 10,
    fetch_k         : int   = 50,
    alpha           : float = 0.70,
    beta            : float = 0.20,
    lambda_mmr      : float = 0.60,
    recency_boost   : float = 0.15,
    half_life_years : float = 3.0,
) -> pd.DataFrame:
    query_vec        = embeddings[query_idx].reshape(1, -1).astype(np.float32)
    sem_scores, idxs = faiss_index.search(query_vec, fetch_k + 1)

    rows = []
    for i, s in zip(idxs[0], sem_scores[0]):
        idx = int(i)
        if idx == query_idx:
            continue
        graph_s = float(df.iloc[idx]["graph_score"])
        rows.append({
            "cand_idx"       : idx,
            "paper_id"       : df.iloc[idx]["paper_id"],
            "title"          : df.iloc[idx]["title"],
            "categories"     : df.iloc[idx]["categories"],
            "date"           : df.iloc[idx]["date"],
            "graph_pagerank" : df.iloc[idx]["graph_pagerank"],
            "relevance_score": round(alpha * float(s) + beta * graph_s, 4),
        })

    candidates   = pd.DataFrame(rows)
    cand_indices = candidates["cand_idx"].tolist()

    mmr_results   = maximal_marginal_relevance(
        query_vec.flatten(), cand_indices, embeddings,
        top_n=top_n * 2, lambda_param=lambda_mmr
    )
    mmr_idx_order = [r[0] for r in mmr_results]
    idx_to_row    = {row["cand_idx"]: row for _, row in candidates.iterrows()}
    reranked      = pd.DataFrame(
        [idx_to_row[i] for i in mmr_idx_order if i in idx_to_row]
    ).head(top_n * 2).reset_index(drop=True)

    boosted = apply_recency_boost(reranked, half_life_years, recency_boost)
    boosted = boosted.head(top_n).reset_index(drop=True)
    final   = attach_explanations(query_idx, boosted, df, G)

    return final[[
        "paper_id", "title", "categories", "date",
        "relevance_score", "recency", "final_score", "why_recommended"
    ]]

In [10]:
print("\n" + "=" * 60)
print("FULL PIPELINE TEST")
print("=" * 60)

sample_idx = 42
print(f"\nQuery: {df.iloc[sample_idx]['title']}\n")

final_recs = full_recommendation_pipeline(
    sample_idx, embeddings, faiss_index, df, G, top_n=10
)
pd.set_option("display.max_colwidth", 60)
print(final_recs[["title", "final_score", "recency", "why_recommended"]].to_string())


print("\n" + "=" * 60)
print("MMR LAMBDA COMPARISON")
print("=" * 60)

rec_indices = list(range(sample_idx + 1, sample_idx + 51))
for lam in [1.0, 0.7, 0.5, 0.3]:
    mmr_out  = maximal_marginal_relevance(
        embeddings[sample_idx], rec_indices, embeddings, top_n=10, lambda_param=lam
    )
    out_idxs = [r[0] for r in mmr_out]
    out_df   = df.iloc[out_idxs][["title", "categories", "graph_pagerank"]].copy()
    ild      = intra_list_diversity(out_df, embeddings, out_idxs)
    cat_div  = category_diversity(out_df)
    print(f"\n  λ={lam}  ILD={ild}  CatDiv={cat_div}")
    for r in mmr_out[:3]:
        print(f"    [{r[1]:.4f}] {df.iloc[r[0]]['title'][:70]}")


FULL PIPELINE TEST

Query: The Parameter-Less Self-Organizing Map algorithm

                                                                                                                       title  final_score  recency                   why_recommended
0                                                      Traitement Des Donnees Manquantes Au Moyen De L'Algorithme De Kohonen       0.8518   0.0120                 Trending in cs.NE
1                                                                      A Study in a Hybrid Centralised-Swarm Agent Community       0.6228   0.0122  Frequently cited with your paper
2                             Fuzzy Artmap and Neural Network Approach to Online Processing of Inputs\n  with Missing Values       0.4182   0.0122                 Trending in cs.AI
3  Exploiting Heavy Tails in Training Times of Multilayer Perceptrons: A\n  Case Study with the UCI Thyroid Disease Database       0.4110   0.0120                 Trending in cs.NE
4                

In [11]:
np.random.seed(42)
fairness_report      = fairness_audit(df, embeddings, faiss_index, sample_size=110, top_n=10)
all_metrics["fairness"] = fairness_report

print("\n" + "=" * 60)
print("DIVERSITY — Before vs After MMR")
print("=" * 60)

np.random.seed(42)
sample_queries = np.random.choice(len(df), size=50, replace=False)
before_ild, after_ild = [], []
before_cd,  after_cd  = [], []

for qidx in sample_queries:
    qvec        = embeddings[qidx].reshape(1, -1).astype(np.float32)
    _, raw_idxs = faiss_index.search(qvec, 11)
    raw_idxs    = [int(i) for i in raw_idxs[0] if i != qidx][:10]

    mmr_out  = maximal_marginal_relevance(
        embeddings[qidx], raw_idxs, embeddings, top_n=10, lambda_param=0.60
    )
    mmr_idxs = [r[0] for r in mmr_out]

    raw_df = df.iloc[raw_idxs]
    mmr_df = df.iloc[mmr_idxs]

    before_ild.append(intra_list_diversity(raw_df, embeddings, raw_idxs))
    after_ild.append(intra_list_diversity(mmr_df, embeddings, mmr_idxs))
    before_cd.append(category_diversity(raw_df))
    after_cd.append(category_diversity(mmr_df))

diversity_metrics = {
    "before_mmr_ild" : round(float(np.mean(before_ild)), 4),
    "after_mmr_ild"  : round(float(np.mean(after_ild)),  4),
    "before_mmr_cd"  : round(float(np.mean(before_cd)),  4),
    "after_mmr_cd"   : round(float(np.mean(after_cd)),   4),
    "ild_improvement": round(float(np.mean(after_ild)) - float(np.mean(before_ild)), 4),
    "cd_improvement" : round(float(np.mean(after_cd))  - float(np.mean(before_cd)),  4),
}

print(f"\n  {'Metric':<30} {'Before':>10} {'After':>10} {'Delta':>10}")
print("  " + "-" * 62)
for label, before_key, after_key, delta_key in [
    ("Intra-List Diversity", "before_mmr_ild", "after_mmr_ild", "ild_improvement"),
    ("Category Diversity",   "before_mmr_cd",  "after_mmr_cd",  "cd_improvement"),
]:
    b = diversity_metrics[before_key]
    a = diversity_metrics[after_key]
    d = diversity_metrics[delta_key]
    print(f"  {label:<30} {b:>10} {a:>10} {'▲' if d>0 else '▼'} {abs(d):>8}")

all_metrics["diversity"] = diversity_metrics


FAIRNESS AUDIT REPORT

  Corpus mean PageRank      : 0.00909091
  Rec mean PageRank         : 0.00941901
  Popularity bias ratio     : 1.036x
    (>1.0 = over-recommends popular papers)

  Top-10% PageRank exposure : 10.7%
    (% of recs from top-10% most cited papers)

  Recency exposure (2yr)    : 0.0%
    (% of recs from last 2 years)

DIVERSITY — Before vs After MMR

  Metric                             Before      After      Delta
  --------------------------------------------------------------
  Intra-List Diversity               0.7316     0.7316 ▼      0.0
  Category Diversity                 0.4739     0.4739 ▼      0.0


In [12]:
print("\n" + "=" * 60)
print("SAVING ARTIFACTS")
print("=" * 60)

sample_recs = full_recommendation_pipeline(
    42, embeddings, faiss_index, df, G, top_n=10
)
sample_recs.to_parquet("/kaggle/working/sample_recommendations.parquet", index=False)
print("Saved: sample_recommendations.parquet")

with open("/kaggle/working/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print("Saved: all_metrics.json")

pipeline_config = {
    "alpha"          : 0.70,
    "beta"           : 0.20,
    "lambda_mmr"     : 0.60,
    "recency_boost"  : 0.15,
    "half_life_years": 3.0,
    "fetch_k"        : 50,
    "top_n"          : 10,
}
with open("/kaggle/working/pipeline_config.json", "w") as f:
    json.dump(pipeline_config, f, indent=2)
print("Saved: pipeline_config.json")


SAVING ARTIFACTS
Saved: sample_recommendations.parquet
Saved: all_metrics.json
Saved: pipeline_config.json
